<a href="https://colab.research.google.com/github/TANUSHRI-BHISE/deep-learning/blob/main/cat_vs_dog_after_augumentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset

Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
100% 775M/775M [00:04<00:00, 192MB/s]



In [3]:
import zipfile
zip_ref=zipfile.ZipFile('/content/dog-and-cat-classification-dataset.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [4]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout

In [6]:
#create folders
import os

base_dir = "/content/dataset"

folders = [
    "train/cat", "train/dog",
    "validation/cat", "validation/dog",
    "test/cat", "test/dog"
]

for f in folders:
    os.makedirs(os.path.join(base_dir, f), exist_ok=True)

In [7]:
import shutil
import random

def split_fixed(source, train_dir, val_dir, test_dir, train_size=1000):
    files = os.listdir(source)
    random.shuffle(files)

    # Select exactly 1000 for training
    train_files = files[:train_size]

    remaining = files[train_size:]

    # Split remaining into validation & test (50-50)
    split_point = len(remaining) // 2
    val_files = remaining[:split_point]
    test_files = remaining[split_point:]

    for file in train_files:
        shutil.copy(os.path.join(source, file), train_dir)

    for file in val_files:
        shutil.copy(os.path.join(source, file), val_dir)

    for file in test_files:
        shutil.copy(os.path.join(source, file), test_dir)

In [8]:
split_fixed("/content/PetImages/Cat",
            "/content/dataset/train/cat",
            "/content/dataset/validation/cat",
            "/content/dataset/test/cat")

split_fixed("/content/PetImages/Dog",
            "/content/dataset/train/dog",
            "/content/dataset/validation/dog",
            "/content/dataset/test/dog")

In [10]:
from PIL import Image
import os

def clean_folder(folder):
    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        # ✅ Skip directories
        if os.path.isdir(path):
            continue

        try:
            img = Image.open(path)
            img.verify()
        except:
            print("Removing:", path)
            os.remove(path)

In [11]:
folders = [
    "/content/dataset/train/cat",
    "/content/dataset/train/dog",
    "/content/dataset/validation/cat",
    "/content/dataset/validation/dog",
    "/content/dataset/test/cat",
    "/content/dataset/test/dog"
]

for folder in folders:
    clean_folder(folder)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [12]:
import os

print("Train Cats:", len(os.listdir("/content/dataset/train/cat")))
print("Train Dogs:", len(os.listdir("/content/dataset/train/dog")))

Train Cats: 1000
Train Dogs: 1000


In [14]:
import os
import numpy as np
import cv2

def load_data(folder):
    images = []
    labels = []

    for label_name in ["cat", "dog"]:
        path = os.path.join(folder, label_name)

        label = 0 if label_name == "cat" else 1

        for file in os.listdir(path):
            img_path = os.path.join(path, file)

            try:
                img = cv2.imread(img_path)

                # Skip if image not loaded
                if img is None:
                    continue

                img = cv2.resize(img, (150, 150))
                img = img / 255.0

                images.append(img)
                labels.append(label)

            except:
                continue

    return np.array(images), np.array(labels)

In [16]:
#create cnn model
model=Sequential()
model.add(Conv2D(32,kernel_size=(3,3),activation='relu',input_shape=(150,150,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2))

model.add(Conv2D(32,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2))

model.add(Conv2D(64,kernel_size=(3,3),activation='relu',))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,))

model.add(Flatten())

model.add(Dense(64,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
from tensorflow.keras.utils import image_dataset_from_directory

In [18]:
train_ds = image_dataset_from_directory(
    "/content/dataset/train",
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary'
)

Found 2000 files belonging to 2 classes.


In [19]:
val_ds = image_dataset_from_directory(
    "/content/dataset/validation",
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary'
)

Found 11498 files belonging to 2 classes.


In [20]:
test_ds = image_dataset_from_directory(
    "/content/dataset/test",
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary'
)

Found 11500 files belonging to 2 classes.


In [21]:
import tensorflow as tf

train_ds = train_ds.map(lambda x, y: (x/255.0, y))
val_ds = val_ds.map(lambda x, y: (x/255.0, y))
test_ds = test_ds.map(lambda x, y: (x/255.0, y))

In [24]:
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())
test_ds = test_ds.apply(tf.data.experimental.ignore_errors())

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [25]:
history = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

Epoch 1/5
     61/Unknown 59s 942ms/step - accuracy: 0.5326 - loss: 0.7023

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


61/61 ━━━━━━━━━━━━━━━━━━━━ 164s 3s/step - accuracy: 0.5496 - loss: 0.7007 - val_accuracy: 0.6077 - val_loss: 0.6792
Epoch 2/5
61/61 ━━━━━━━━━━━━━━━━━━━━ 162s 3s/step - accuracy: 0.6018 - loss: 0.6689 - val_accuracy: 0.6454 - val_loss: 0.6307
Epoch 3/5
61/61 ━━━━━━━━━━━━━━━━━━━━ 173s 3s/step - accuracy: 0.6534 - loss: 0.6299 - val_accuracy: 0.6760 - val_loss: 0.6003
Epoch 4/5
61/61 ━━━━━━━━━━━━━━━━━━━━ 161s 3s/step - accuracy: 0.6875 - loss: 0.5929 - val_accuracy: 0.6360 - val_loss: 0.6109
Epoch 5/5
61/61 ━━━━━━━━━━━━━━━━━━━━ 164s 3s/step - accuracy: 0.7076 - loss: 0.5743 - val_accuracy: 0.5283 - val_loss: 1.6539


In [26]:

from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [29]:
batch_size=16
train_datagen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,#angle
    shear_range=0.2,#distortion
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='reflect'#we can also use constant to fill black

)

test_datagen=ImageDataGenerator(
    rescale=1./255)

train_genrator=train_datagen.flow_from_directory(
    '/content/dataset/test',
    target_size=(150,150),
    batch_size=batch_size,
    class_mode='binary'
)

validation_genrator=test_datagen.flow_from_directory(
    '/content/dataset/validation',
    target_size=(150,150),
    batch_size=batch_size,
    class_mode='binary'
)

Found 11500 images belonging to 2 classes.
Found 11498 images belonging to 2 classes.


In [34]:
import shutil

shutil.rmtree("/content/dataset")

In [35]:
import os

folders = [
    "train/cat", "train/dog",
    "validation/cat", "validation/dog",
    "test/cat", "test/dog"
]

for f in folders:
    os.makedirs(os.path.join("/content/dataset", f), exist_ok=True)

In [36]:
from PIL import Image

def clean_folder(folder):
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        try:
            img = Image.open(path)
            img.verify()
        except:
            os.remove(path)

clean_folder("/content/PetImages/Cat")
clean_folder("/content/PetImages/Dog")

In [37]:
import shutil
import random
import os

def split_exact(source, train_dir, val_dir, test_dir):
    files = os.listdir(source)
    random.shuffle(files)

    test_files = files[:1000]        # exactly 1000
    val_files = files[1000:1500]     # exactly 500
    train_files = files[1500:]       # remaining

    for file in test_files:
        shutil.copy(os.path.join(source, file), test_dir)

    for file in val_files:
        shutil.copy(os.path.join(source, file), val_dir)

    for file in train_files:
        shutil.copy(os.path.join(source, file), train_dir)

In [38]:
split_exact("/content/PetImages/Cat",
            "/content/dataset/train/cat",
            "/content/dataset/validation/cat",
            "/content/dataset/test/cat")

split_exact("/content/PetImages/Dog",
            "/content/dataset/train/dog",
            "/content/dataset/validation/dog",
            "/content/dataset/test/dog")

In [39]:
import os

print("Test Cats:", len(os.listdir("/content/dataset/test/cat")))
print("Test Dogs:", len(os.listdir("/content/dataset/test/dog")))

print("Val Cats:", len(os.listdir("/content/dataset/validation/cat")))
print("Val Dogs:", len(os.listdir("/content/dataset/validation/dog")))

Test Cats: 1000
Test Dogs: 1000
Val Cats: 500
Val Dogs: 500


In [40]:
#create cnn model
model=Sequential()
model.add(Conv2D(32,kernel_size=(3,3),activation='relu',input_shape=(150,150,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2))

model.add(Conv2D(32,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2))

model.add(Conv2D(64,kernel_size=(3,3),activation='relu',))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,))

model.add(Flatten())

model.add(Dense(64,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [41]:

from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [42]:
batch_size=16
train_datagen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,#angle
    shear_range=0.2,#distortion
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='reflect'#we can also use constant to fill black

)

test_datagen=ImageDataGenerator(
    rescale=1./255)

train_genrator=train_datagen.flow_from_directory(
    '/content/dataset/test',
    target_size=(150,150),
    batch_size=batch_size,
    class_mode='binary'
)

validation_genrator=test_datagen.flow_from_directory(
    '/content/dataset/validation',
    target_size=(150,150),
    batch_size=batch_size,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [43]:
#create cnn model
model=Sequential()
model.add(Conv2D(32,kernel_size=(3,3),activation='relu',input_shape=(150,150,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2))

model.add(Conv2D(32,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2))

model.add(Conv2D(64,kernel_size=(3,3),activation='relu',))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,))

model.add(Flatten())

model.add(Dense(64,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

In [47]:
model.fit(
    train_genrator,
    steps_per_epoch=2000,#batch size,
    epochs=5,
    validation_data=validation_genrator,
    validation_steps=800
)

Epoch 1/5
 125/2000 ━━━━━━━━━━━━━━━━━━━━ 18:00 576ms/step - accuracy: 0.5186 - loss: 0.7064

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


2000/2000 ━━━━━━━━━━━━━━━━━━━━ 84s 41ms/step - accuracy: 0.5225 - loss: 0.6990 - val_accuracy: 0.5060 - val_loss: 0.6789
Epoch 2/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 81s 40ms/step - accuracy: 0.5705 - loss: 0.6815 - val_accuracy: 0.6480 - val_loss: 0.6405
Epoch 3/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 81s 40ms/step - accuracy: 0.6160 - loss: 0.6633 - val_accuracy: 0.6320 - val_loss: 0.6280
Epoch 4/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 81s 40ms/step - accuracy: 0.6335 - loss: 0.6533 - val_accuracy: 0.6450 - val_loss: 0.6145
Epoch 5/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 80s 40ms/step - accuracy: 0.6320 - loss: 0.6358 - val_accuracy: 0.6560 - val_loss: 0.6031
